In [1]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
import torch
import seaborn as sns
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import ast
import re


In [2]:
2+2

4

In [3]:
import torch
from transformers import AutoConfig, AutoModel

In [6]:
import torch
from transformers import AutoModel, AutoConfig

MODEL_ID = "apiantonio/vjepa2.1-vit-base-384"

# 1. Load config and model directly using local Hugging Face cache
# (Since trust_remote_code=True is required for custom model architectures like V-JEPA 2.1)
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval()

# 2. Inspect Config Architecture Features
print("=== Model Architecture Features ===")
print(f"Model Type:            {getattr(config, 'model_type', 'N/A')}")
print(f"Hidden Dimension (D):  {config.hidden_size}")
print(f"Image Resolution:      {getattr(config, 'image_size', 384)} x {getattr(config, 'image_size', 384)}")
print(f"Patch Size:            {config.patch_size} x {config.patch_size}")
print(f"Tubelet Size (Frames): {getattr(config, 'tubelet_size', 2)}")
print(f"Num Layers / Heads:    {config.num_hidden_layers} layers / {config.num_attention_heads} heads")

# Calculate spatial and temporal patch counts
spatial_patches = (getattr(config, "image_size", 384) // config.patch_size) ** 2
print(f"Patches per Frame:     {spatial_patches} (from (384/16)^2)")

# 3. Test Forward Pass to Verify Input & Output Dimensions
# Expected Input Shape: (Batch, Channels, Frames, Height, Width)
batch_size = 1
channels = 3
frames = 16  # Standard clip length (must be divisible by tubelet_size=2)
height, width = 384, 384

dummy_input = torch.randn(batch_size, channels, frames, height, width)

print("\n=== Input / Output Inspection ===")
print(
    f"Required Input Shape: [Batch={batch_size}, Channels={channels}, Frames={frames}, Height={height}, Width={width}]"
)

with torch.no_grad():
    outputs = model(dummy_input)

# Check output structure
if hasattr(outputs, "last_hidden_state"):
    output_tensor = outputs.last_hidden_state
else:
    output_tensor = outputs[0]

print(f"Output Tensor Shape:  {list(output_tensor.shape)}")
print(f"  - Batch Size:        {output_tensor.shape[0]}")
print(
    f"  - Sequence Tokens:   {output_tensor.shape[1]} (Computed as: (16 frames / 2 tubelet_size) * 576 spatial patches)"
)
print(f"  - Feature Dim:       {output_tensor.shape[2]}")

I0000 00:00:1791900415.649166 1606347 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


=== Model Architecture Features ===
Model Type:            vjepa21
Hidden Dimension (D):  768
Image Resolution:      384 x 384
Patch Size:            16 x 16
Tubelet Size (Frames): 2
Num Layers / Heads:    12 layers / 12 heads
Patches per Frame:     576 (from (384/16)^2)

=== Input / Output Inspection ===
Required Input Shape: [Batch=1, Channels=3, Frames=16, Height=384, Width=384]
Output Tensor Shape:  [1, 4608, 768]
  - Batch Size:        1
  - Sequence Tokens:   4608 (Computed as: (16 frames / 2 tubelet_size) * 576 spatial patches)
  - Feature Dim:       768
